# Uncertainty-Aware Clarification Agent — Results Analysis

This notebook loads the evaluation results and generates analysis plots.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

RESULTS_DIR = os.path.join("..", "results")

In [ ]:
# Load raw results
with open(os.path.join(RESULTS_DIR, "raw_results.json"), "r") as f:
    results = json.load(f)

df = pd.DataFrame(results)
print(f"Total records: {len(df)}")
df.head()

In [ ]:
# Overall metrics
metrics = pd.read_csv(os.path.join(RESULTS_DIR, "metrics.csv"))
metrics

In [ ]:
# Per-domain breakdown
domain_metrics = pd.read_csv(os.path.join(RESULTS_DIR, "domain_metrics.csv"))
domain_metrics

In [ ]:
# Uncertainty distribution for the uncertainty agent
ua = df[df["system"] == "uncertainty_agent"]

fig, ax = plt.subplots(figsize=(10, 6))
ambig = ua[ua["expected_label"] == "ambiguous"]["uncertainty_score"]
clear = ua[ua["expected_label"] == "clear"]["uncertainty_score"]

ax.hist(ambig, bins=15, alpha=0.6, label="Ambiguous", color="#e74c3c")
ax.hist(clear, bins=15, alpha=0.6, label="Clear", color="#2ecc71")
ax.axvline(x=0.4, color="black", linestyle="--", linewidth=1.5, label="Threshold (0.4)")
ax.set_xlabel("Uncertainty Score", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title("Uncertainty Score Distribution: Clear vs Ambiguous", fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Decision accuracy comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

systems = metrics["System"].tolist()
x = np.arange(len(systems))
colors = ["#3498db", "#e67e22", "#9b59b6"]

# Accuracy
axes[0].bar(x, metrics["Decision Accuracy (%)"], color=colors)
axes[0].set_title("Decision Accuracy")
axes[0].set_ylabel("%")
axes[0].set_xticks(x)
axes[0].set_xticklabels(systems, rotation=15)
axes[0].set_ylim(0, 105)

# Unnecessary clarification
axes[1].bar(x, metrics["Unnecessary Clarification (%)"], color=colors)
axes[1].set_title("Unnecessary Clarification Rate")
axes[1].set_ylabel("%")
axes[1].set_xticks(x)
axes[1].set_xticklabels(systems, rotation=15)
axes[1].set_ylim(0, 105)

# Avg turns
axes[2].bar(x, metrics["Avg Turns"], color=colors)
axes[2].set_title("Average Turns")
axes[2].set_ylabel("Turns")
axes[2].set_xticks(x)
axes[2].set_xticklabels(systems, rotation=15)
axes[2].set_ylim(0, 3)

plt.suptitle("System Comparison", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Uncertainty vs query length scatter
ua_copy = ua.copy()
ua_copy["query_length"] = ua_copy["query"].str.len()

fig, ax = plt.subplots(figsize=(10, 6))
colors_map = ua_copy["expected_label"].map({"ambiguous": "#e74c3c", "clear": "#2ecc71"})
ax.scatter(ua_copy["query_length"], ua_copy["uncertainty_score"], c=colors_map, alpha=0.7, s=80, edgecolors="gray")
ax.set_xlabel("Query Length (characters)", fontsize=12)
ax.set_ylabel("Uncertainty Score", fontsize=12)
ax.set_title("Uncertainty Score vs Query Length", fontsize=14)

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#e74c3c", markersize=10, label="Ambiguous"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#2ecc71", markersize=10, label="Clear"),
]
ax.legend(handles=legend_elements, fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Per-domain accuracy heatmap
fig, ax = plt.subplots(figsize=(8, 5))
domain_metrics_sorted = domain_metrics.sort_values("Decision Accuracy (%)", ascending=True)
ax.barh(domain_metrics_sorted["Domain"], domain_metrics_sorted["Decision Accuracy (%)"], color="#3498db")
ax.set_xlabel("Decision Accuracy (%)")
ax.set_title("Uncertainty Agent: Per-Domain Accuracy")
ax.set_xlim(0, 105)
plt.tight_layout()
plt.show()